# CALCULADORA ONSHORE vs OFFSHORE

Introduir l'Excel "Excel_Onshore_vs_Offshore.xlsx" i donar-li al botó d'Exectuar per a obtenir els resultats. Prèviament s'ha hagut d'executar la primera cel·la, la qual descarrega les llibreries necessàries per a executar el codi d'abaix.

LLIBRERIA NECESSÀRIA A EXECUTAR EL CODI

In [3]:
!apt-get update -qq
!apt-get install -y weasyprint > /dev/null
!apt-get install -y libpango-1.0-0 libpangoft2-1.0-0 > /dev/null
!pip install weasyprint --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%


Document PDF + EXCEL

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.stats import weibull_min
from scipy.optimize import newton
import os
import base64
import zipfile
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as OpenImage
from openpyxl.styles import Font, Border, Side, PatternFill
import warnings

# Amagar avisos de validació de dades de l'Excel
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

try:
    from google.colab import files
    colab_env = True
except ImportError:
    colab_env = False

import logging
# Silenciar els missatges d'informació de creació del PDF
logging.getLogger('fontTools').setLevel(logging.WARNING)
logging.getLogger('weasyprint').setLevel(logging.WARNING)

# =========================================================
# 1. CARREGAR DADES
# =========================================================
file_name = 'Excel_Onshore_vs_Offshore.xlsx'
df_calc = pd.read_excel(file_name, sheet_name='Calculadora')
df_bbdd = pd.read_excel(file_name, sheet_name='BBDD')

ppa_onshore  = float(df_calc.iloc[3, 11])
ppa_offshore = float(df_calc.iloc[7, 11])

df_clean = df_calc.dropna(subset=['Temps', 'v_vent_onshore', 'v_vent_offshore']).copy()
df_clean['datetime'] = pd.to_datetime(df_clean['Temps'], errors='coerce')
df_clean['MO'] = df_clean['datetime'].dt.month

# =========================================================
# 2. MODELS TÈCNICS I PARÀMETRES
# =========================================================
p_on_ref = df_bbdd.iloc[:, 21].dropna().values
v_on_ref = df_bbdd.iloc[:, 22].dropna().values
f_interp_on = interp1d(v_on_ref, p_on_ref, kind='linear', bounds_error=False, fill_value=(0, 0))
p_park_mw_on = max(p_on_ref) / 1000

p_off_ref = df_bbdd.iloc[:, 23].dropna().values
v_off_ref = df_bbdd.iloc[:, 24].dropna().values
f_interp_off = interp1d(v_off_ref, p_off_ref, kind='linear', bounds_error=False, fill_value=(0, 0))
p_park_mw_off = max(p_off_ref) / 1000

opex_off_rate   = float(df_bbdd.iloc[:, 40].dropna().iloc[0])
opex_on_rate    = float(df_bbdd.iloc[:, 41].dropna().iloc[0])
capex_on_rate   = float(df_bbdd.iloc[:, 42].dropna().iloc[0])
capex_off_rate  = float(df_bbdd.iloc[:, 43].dropna().iloc[0])
taxa_descompte  = float(df_bbdd.iloc[:, 45].dropna().iloc[0])
vida_util       = 25
inflacio        = float(df_bbdd.iloc[:, 46].dropna().iloc[0])

f_rendiment_on  = float(df_bbdd.iloc[0, 44])
f_rendiment_off = float(df_bbdd.iloc[3, 44])

tax_rate   = float(df_bbdd['Impost de societats (%)'].dropna().iloc[0])
grid_fee   = float(df_bbdd['Grid Connection Fee (%)'].dropna().iloc[0])
peatge_mwh = float(df_bbdd['Representacio mercat (€/MWh)'].dropna().iloc[0])
anys_amort_fiscal = int(df_bbdd.iloc[0, 56])
degradacio = float(df_bbdd.iloc[0, 39])
incertesa  = float(df_bbdd.iloc[0, 51])
z_p90      = float(df_bbdd.iloc[0, 52])


try:
    val_off = pd.to_numeric(df_bbdd.iloc[0, 66:71], errors='coerce').fillna(0).values
    if sum(val_off) > 1.5: val_off = val_off / 100
    pay_sch_off = list(val_off)
except: pay_sch_off = [0.1, 0.2, 0.3, 0.1, 0.3]

try:
    val_on = pd.to_numeric(df_bbdd.iloc[0, 73:76], errors='coerce').fillna(0).values
    if sum(val_on) > 1.5: val_on = val_on / 100
    pay_sch_on = list(val_on)
except: pay_sch_on = [0.1, 0.3, 0.6]

try: devex_off_rate = float(df_bbdd.iloc[0, 77])
except: devex_off_rate = 0.15

try: devex_on_rate = float(df_bbdd.iloc[0, 78])
except: devex_on_rate = 0.05

try: decom_off_rate = float(df_bbdd.iloc[0, 80])
except: decom_off_rate = 0.3

try: decom_on_rate = float(df_bbdd.iloc[0, 81])
except: decom_on_rate = 0.15

pre_op_on = len(pay_sch_on)
pre_op_off = len(pay_sch_off)

# =========================================================
# 3. SIMULACIÓ ENERGÈTICA
# =========================================================
df_clean['Gross_Energy_ON_MWh'] = df_clean['v_vent_onshore'].apply(lambda v: float(f_interp_on(v)) if pd.notna(v) and v <= 25 else 0) / 1000
df_clean['Gross_Energy_OFF_MWh'] = df_clean['v_vent_offshore'].apply(lambda v: float(f_interp_off(v)) if pd.notna(v) and v <= 25 else 0) / 1000

num_anys_estudi = len(df_clean) / 8760.0
prod_on_anual_bruta = df_clean['Gross_Energy_ON_MWh'].sum() / num_anys_estudi
prod_off_anual_bruta = df_clean['Gross_Energy_OFF_MWh'].sum() / num_anys_estudi

prod_on_p50 = prod_on_anual_bruta * f_rendiment_on
prod_off_p50 = prod_off_anual_bruta * f_rendiment_off

# =========================================================
# 4. CÀLCULS FINANCERS I PROJECT FINANCE
# =========================================================
h_eq_on, h_eq_off = prod_on_p50 / p_park_mw_on, prod_off_p50 / p_park_mw_off
fc_on, fc_off = (prod_on_p50 / (p_park_mw_on * 8760)) * 100, (prod_off_p50 / (p_park_mw_off * 8760)) * 100

fc_gross_on = (prod_on_anual_bruta / (p_park_mw_on * 8760)) * 100
fc_gross_off = (prod_off_anual_bruta / (p_park_mw_off * 8760)) * 100

fc_p90_on = ((prod_on_p50*(1-z_p90*incertesa)) / (p_park_mw_on * 8760)) * 100
fc_p90_off = ((prod_off_p50*(1-z_p90*incertesa)) / (p_park_mw_off * 8760)) * 100

cap_on_total  = ((p_park_mw_on * capex_on_rate) * (1 + grid_fee)) + (p_park_mw_on * devex_on_rate)
cap_off_total = ((p_park_mw_off * capex_off_rate) * (1 + grid_fee)) + (p_park_mw_off * devex_off_rate)

amort_on_anual = cap_on_total / anys_amort_fiscal
amort_off_anual = cap_off_total / anys_amort_fiscal

decom_cost_on = p_park_mw_on * decom_on_rate
decom_cost_off = p_park_mw_off * decom_off_rate

ppa_on_adj = ppa_onshore * (1 + inflacio)**pre_op_on
ppa_off_adj = ppa_offshore * (1 + inflacio)**pre_op_off

op_on_base_adj = (p_park_mw_on * opex_on_rate) * (1 + inflacio)**pre_op_on
op_off_base_adj = (p_park_mw_off * opex_off_rate) * (1 + inflacio)**pre_op_off

rev_on_1 = prod_on_p50 * ppa_on_adj / 1e6
rev_off_1 = prod_off_p50 * ppa_off_adj / 1e6

anys_idx = np.arange(1, vida_util + 1)
rev_on_l = [rev_on_1 * ((1-degradacio)*(1+inflacio))**(t-1) for t in anys_idx]
rev_off_l = [rev_off_1 * ((1-degradacio)*(1+inflacio))**(t-1) for t in anys_idx]

op_on_l = [(op_on_base_adj*(1+inflacio)**(t-1)) + (prod_on_p50*(1-degradacio)**(t-1)*peatge_mwh/1e6) for t in anys_idx]
op_off_l = [(op_off_base_adj*(1+inflacio)**(t-1)) + (prod_off_p50*(1-degradacio)**(t-1)*peatge_mwh/1e6) for t in anys_idx]

# =========================================================
# 5. TAULA ECONÒMICA
# =========================================================
def generar_taula_economica(cap_total, pay_sch, decom_cost, rev_l, op_l, amort, tax_r, taxa, anys, anys_a):
    pre_op = len(pay_sch)
    total_y = pre_op + anys
    any_l = list(range(-pre_op, 0)) + list(range(1, anys + 1))

    rev_full = [0.0]*pre_op + list(rev_l)
    op_full = [0.0]*pre_op + list(op_l)
    op_full[-1] += decom_cost

    ebitda, amort_l, ebit, imp, cf = [0.0]*total_y, [0.0]*total_y, [0.0]*total_y, [0.0]*total_y, [0.0]*total_y
    carryforward = 0.0

    for i in range(total_y):
        t_op = i - pre_op + 1
        if i < pre_op: cf[i] = -cap_total * pay_sch[i]
        else:
            ebitda[i] = rev_full[i] - op_full[i]
            amort_l[i] = amort if t_op <= anys_a else 0
            ebit[i] = ebitda[i] - amort_l[i]

            base_imposable = ebit[i] + carryforward
            if base_imposable > 0:
                imp[i] = base_imposable * tax_r
                carryforward = 0.0
            else:
                imp[i] = 0.0
                carryforward = base_imposable

            cf[i] = ebitda[i] - imp[i]

    cf_desc = [cf[i] / (1 + taxa)**i for i in range(total_y)]
    return pd.DataFrame({'Any': any_l, 'Ingressos (M€)': rev_full, 'OPEX Total (M€)': op_full, 'EBITDA (M€)': ebitda, 'Amortització (M€)': amort_l, 'EBIT (M€)': ebit, 'Impostos (M€)': imp, 'Flux de Caixa Simple (M€)': cf, 'Acumulat Simple (M€)': np.cumsum(cf).tolist(), 'Flux de Caixa Descomptat (M€)': cf_desc, 'Acumulat Descomptat (M€)': np.cumsum(cf_desc).tolist()})

df_econ_on = generar_taula_economica(cap_on_total, pay_sch_on, decom_cost_on, rev_on_l, op_on_l, amort_on_anual, tax_rate, taxa_descompte, vida_util, anys_amort_fiscal)
df_econ_off = generar_taula_economica(cap_off_total, pay_sch_off, decom_cost_off, rev_off_l, op_off_l, amort_off_anual, tax_rate, taxa_descompte, vida_util, anys_amort_fiscal)

columnes_exactes = ['Any', 'Ingressos (M€)', 'OPEX Total (M€)', 'EBITDA (M€)', 'Amortització (M€)', 'EBIT (M€)', 'Impostos (M€)', 'Flux de Caixa Simple (M€)', 'Acumulat Simple (M€)', 'Flux de Caixa Descomptat (M€)', 'Acumulat Descomptat (M€)']
df_analisi = pd.concat([pd.DataFrame([['ONSHORE']+[np.nan]*10], columns=columnes_exactes), df_econ_on, pd.DataFrame([[np.nan]*11], columns=columnes_exactes), pd.DataFrame([['OFFSHORE']+[np.nan]*10], columns=columnes_exactes), df_econ_off], ignore_index=True)

def calc_lcoe(cap_total, pay_sch, op_full, prod_p50, taxa, pre_op):
    tot_y = pre_op + vida_util
    c_desc = sum((cap_total * (pay_sch[i] if i < pre_op else 0) + op_full[i]) / (1+taxa)**i for i in range(tot_y))
    g_desc = sum((prod_p50 * (1-degradacio)**(i-pre_op)) / (1+taxa)**i for i in range(pre_op, tot_y))
    return (c_desc * 1e6) / g_desc

lcoe_on = calc_lcoe(cap_on_total, pay_sch_on, df_econ_on['OPEX Total (M€)'].values, prod_on_p50, taxa_descompte, pre_op_on)
lcoe_off = calc_lcoe(cap_off_total, pay_sch_off, df_econ_off['OPEX Total (M€)'].values, prod_off_p50, taxa_descompte, pre_op_off)

def exact_payback(df, col_a, col_f, pre_op):
    pos_idx = df[df[col_a] >= 0].index
    if len(pos_idx) == 0: return ">25"
    any_p_idx = pos_idx.min()
    if any_p_idx == 0: return 0
    fraction = abs(df.loc[any_p_idx-1, col_a]) / df.loc[any_p_idx, col_f] if df.loc[any_p_idx, col_f] != 0 else 0
    val_final = (any_p_idx - pre_op) + fraction
    return ">25" if val_final > 25 else val_final

def calc_tir_full(cf_list):
    def npv_func(r): return sum(cf / (1 + r)**t for t, cf in enumerate(cf_list))
    try: return newton(npv_func, 0.1) * 100
    except: return np.nan

p_s_on, p_s_off = exact_payback(df_econ_on, 'Acumulat Simple (M€)', 'Flux de Caixa Simple (M€)', pre_op_on), exact_payback(df_econ_off, 'Acumulat Simple (M€)', 'Flux de Caixa Simple (M€)', pre_op_off)
p_d_on, p_d_off = exact_payback(df_econ_on, 'Acumulat Descomptat (M€)', 'Flux de Caixa Descomptat (M€)', pre_op_on), exact_payback(df_econ_off, 'Acumulat Descomptat (M€)', 'Flux de Caixa Descomptat (M€)', pre_op_off)

van_on = df_econ_on['Acumulat Descomptat (M€)'].iloc[-1]
van_off = df_econ_off['Acumulat Descomptat (M€)'].iloc[-1]
tir_on = calc_tir_full(df_econ_on['Flux de Caixa Simple (M€)'].values)
tir_off = calc_tir_full(df_econ_off['Flux de Caixa Simple (M€)'].values)

res_data = {
    'Mètrica Analitzada': [
        'Producció Bruta Teòrica (Gross P50) - Mitjana Anual (MWh/any)', 'Producció Neta Esperada (Net P50) - Mitjana Anual (MWh/any)', 'Producció Neta Conservadora (Net P90) - Mitjana Anual (MWh/any)', 'Hores equivalents a plena potència (h)', 'Factor de Capacitat (%) (Net P50)', 'Factor de capacitat (Gross %)', 'Factor de capacitat (Net 10-year P90 %)', 'Ingressos Bruts Anuals - Any 1 (M€/any)', 'OPEX Base (Manteniment Fix) - Any 1 (M€/any)', 'Costos Operatius Totals (OPEX + Peatges) - Any 1 (M€/any)', 'EBITDA (Benefici Brut Operatiu) - Any 1 (M€/any)', 'EBITDA Promig (25 anys) (M€/any)', 'Amortització Lineal Anual (M€/any)', 'EBIT (Benefici abans d\'impostos) - Any 1 (M€/any)', 'Impost de Societats Pagat - Any 1 (M€/any)', 'Benefici Net Anual (Comptable) - Any 1 (M€/any)', 'Flux de Caixa Lliure (Free Cash Flow) - Any 1 (M€/any)', 'Inversió CAPEX + DEVEX Total (M€)', 'Flux de Caixa Net Acumulat (25 anys) (M€)', 'VAN (Valor Actual Net a FID) (M€)', 'TIR (Taxa Interna de Rendiment) (%)', 'LCOE (€/MWh)', 'Payback Simple des d\'Operació (anys)', 'Payback Descomptat des d\'Operació (anys)', 't CO2 equivalent evitats - Any 1', 't CO2 equivalent evitats (Total 25 anys)'
    ],
    'ONSHORE': [
        prod_on_anual_bruta, prod_on_p50, prod_on_p50*(1-z_p90*incertesa), h_eq_on, fc_on, fc_gross_on, fc_p90_on, df_econ_on.loc[pre_op_on, 'Ingressos (M€)'], op_on_base_adj, df_econ_on.loc[pre_op_on, 'OPEX Total (M€)'], df_econ_on.loc[pre_op_on, 'EBITDA (M€)'], df_econ_on['EBITDA (M€)'][pre_op_on:].mean(), amort_on_anual, df_econ_on.loc[pre_op_on, 'EBIT (M€)'], df_econ_on.loc[pre_op_on, 'Impostos (M€)'], df_econ_on.loc[pre_op_on, 'EBIT (M€)'] - df_econ_on.loc[pre_op_on, 'Impostos (M€)'], df_econ_on.loc[pre_op_on, 'Flux de Caixa Simple (M€)'], cap_on_total, df_econ_on['Acumulat Simple (M€)'].iloc[-1], van_on, tir_on, lcoe_on, p_s_on, p_d_on, 0.102*prod_on_p50, sum(0.102*prod_on_p50*(1-degradacio)**(anys_idx-1))
    ],
    'OFFSHORE': [
        prod_off_anual_bruta, prod_off_p50, prod_off_p50*(1-z_p90*incertesa), h_eq_off, fc_off, fc_gross_off, fc_p90_off, df_econ_off.loc[pre_op_off, 'Ingressos (M€)'], op_off_base_adj, df_econ_off.loc[pre_op_off, 'OPEX Total (M€)'], df_econ_off.loc[pre_op_off, 'EBITDA (M€)'], df_econ_off['EBITDA (M€)'][pre_op_off:].mean(), amort_off_anual, df_econ_off.loc[pre_op_off, 'EBIT (M€)'], df_econ_off.loc[pre_op_off, 'Impostos (M€)'], df_econ_off.loc[pre_op_off, 'EBIT (M€)'] - df_econ_off.loc[pre_op_off, 'Impostos (M€)'], df_econ_off.loc[pre_op_off, 'Flux de Caixa Simple (M€)'], cap_off_total, df_econ_off['Acumulat Simple (M€)'].iloc[-1], van_off, tir_off, lcoe_off, p_s_off, p_d_off, 0.102*prod_off_p50, sum(0.102*prod_off_p50*(1-degradacio)**(anys_idx-1))
    ]
}
df_res = pd.DataFrame(res_data)

# =========================================================
# 6. GRÀFIQUES
# =========================================================
img_names = ['g1.png', 'g2.png', 'g3.png', 'g4.png', 'g5.png', 'g6.png']
dates_2025 = df_clean['datetime'][:8760] + pd.DateOffset(years=9)

plt.figure(figsize=(10, 5))
for label, col, color in [('Onshore', 'v_vent_onshore', 'blue'), ('Offshore', 'v_vent_offshore', 'orange')]:
    v_data = df_clean[col].dropna()
    shape, loc, scale = weibull_min.fit(v_data, floc=0)
    plt.hist(v_data, bins=50, density=True, alpha=0.2, color=color)
    plt.plot(np.linspace(0, 35, 200), weibull_min.pdf(np.linspace(0, 35, 200), shape, scale=scale), color=color, lw=2, label=f'{label} (k={shape:.2f}, c={scale:.2f})')
plt.title(f'Distribució de Vent (Sèrie de {num_anys_estudi:.1f} anys)')
plt.xlabel('Velocitat del vent (m/s)')
plt.ylabel('Densitat de probabilitat (%)')
plt.legend(); plt.grid(alpha=0.3); plt.savefig(img_names[0], bbox_inches='tight'); plt.close()

plt.figure(figsize=(12, 4))
plt.plot(dates_2025, df_clean['Gross_Energy_ON_MWh'][:8760], label='Onshore', color='blue', alpha=0.8)
plt.plot(dates_2025, df_clean['Gross_Energy_OFF_MWh'][:8760], label='Offshore', color='orange', alpha=0.6)
plt.title('Producció Energètica bruta horària (Any 2025 - Hourly Gross P50 )')
plt.xlabel('Data (Mesos)')
plt.ylabel('Energia generada (MWh)')
plt.legend(); plt.grid(alpha=0.3); plt.savefig(img_names[1], bbox_inches='tight'); plt.close()

plt.figure(figsize=(12, 4))
plt.plot(dates_2025, df_clean['Gross_Energy_ON_MWh'][:8760] * ppa_onshore, label='Onshore (PPA Base)', color='blue', alpha=0.8)
plt.plot(dates_2025, df_clean['Gross_Energy_OFF_MWh'][:8760] * ppa_offshore, label='Offshore (PPA Base)', color='orange', alpha=0.6)
plt.title('Ingressos Horaris Estimats (Any 2025)')
plt.xlabel('Data (Mesos)')
plt.ylabel('Ingressos (€)')
plt.legend(); plt.grid(alpha=0.3); plt.savefig(img_names[2], bbox_inches='tight'); plt.close()

df_monthly_on = (df_clean.groupby('MO')['Gross_Energy_ON_MWh'].sum() / num_anys_estudi) * f_rendiment_on
df_monthly_off = (df_clean.groupby('MO')['Gross_Energy_OFF_MWh'].sum() / num_anys_estudi) * f_rendiment_off
df_monthly = pd.DataFrame({'Onshore': df_monthly_on, 'Offshore': df_monthly_off})
df_monthly.plot(kind='bar', figsize=(10, 5), color=['blue', 'orange'])
plt.title('Producció Mensual Promig (Any 2025 - Net P50)')
plt.xlabel('Mes de l\'any (Mesos)')
plt.ylabel('Producció neta (MWh)')
plt.xticks(range(12), ['Gen','Feb','Mar','Abr','Mai','Jun','Jul','Ago','Set','Oct','Nov','Des'], rotation=0)
plt.savefig(img_names[3], bbox_inches='tight'); plt.close()

plt.figure(figsize=(12, 6))
plt.plot(df_econ_on['Any'], df_econ_on['Acumulat Simple (M€)'], label='Onshore (Simple)', marker='o', color='blue')
plt.plot(df_econ_off['Any'], df_econ_off['Acumulat Simple (M€)'], label='Offshore (Simple)', marker='s', color='orange')
plt.plot(df_econ_on['Any'], df_econ_on['Acumulat Descomptat (M€)'], label='Onshore (Descomptat)', linestyle='--', marker='x', color='blue')
plt.plot(df_econ_off['Any'], df_econ_off['Acumulat Descomptat (M€)'], label='Offshore (Descomptat)', linestyle='--', marker='+', color='orange')
plt.axhline(0, color='red', linestyle=':')
plt.title('Flux de Caixa Acumulat')
plt.xlabel('Temps d\'operació (Anys)')
plt.ylabel('Flux de Caixa (M€)')
plt.legend(); plt.grid(alpha=0.3); plt.savefig(img_names[4], bbox_inches='tight'); plt.close()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
colors_plot = {'Ingressos': '#2ca02c', 'EBITDA': '#1f77b4', 'EBIT': '#17becf', 'Flux de Caixa': '#9467bd', 'OPEX': '#d62728', 'Amortització': '#ff7f0e', 'Impostos': '#8c564b'}
for ax, df, title in zip([ax1, ax2], [df_econ_on, df_econ_off], ['Radiografia ONSHORE', 'Radiografia OFFSHORE']):
    ax.plot(df['Any'], df['Ingressos (M€)'], label='Ingressos', color=colors_plot['Ingressos'], lw=2.5)
    ax.plot(df['Any'], df['Flux de Caixa Simple (M€)'], label='FC Simple', color=colors_plot['Flux de Caixa'], lw=2.5)
    ax.plot(df['Any'], df['EBITDA (M€)'], label='EBITDA', color=colors_plot['EBITDA'], lw=3.5)
    ax.plot(df['Any'], df['EBIT (M€)'], label='EBIT', color=colors_plot['EBIT'], lw=2, alpha=0.6)
    ax.plot(df['Any'], df['OPEX Total (M€)'], label='OPEX', color=colors_plot['OPEX'], linestyle='--')
    ax.plot(df['Any'], df['Amortització (M€)'], label='Amortització', color=colors_plot['Amortització'], linestyle='--', lw=1.5)
    ax.plot(df['Any'], df['Impostos (M€)'], label='Impostos', color=colors_plot['Impostos'], linestyle='--', lw=1.5)

    ax.set_ylabel('Fluxos Anuals (M€)', fontweight='bold')
    ax.set_xlabel('Anys', fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(alpha=0.3); ax.legend(loc='upper left', ncol=2, fontsize=9)
    ax_acc = ax.twinx()
    ax_acc.plot(df['Any'], df['Acumulat Simple (M€)'], label='Acumulat Simple', color='black', marker='.', markersize=4, lw=1.5, alpha=0.7)
    ax_acc.set_ylabel('Flux Acumulat (M€)', fontweight='bold', color='black')
    ax_acc.axhline(0, color='black', lw=1, linestyle=':')
    ax_acc.legend(loc='lower right', fontsize=9)
plt.xlabel('Temps des de FID (Anys)', fontweight='bold')
plt.tight_layout(pad=3.0, h_pad=2.0); plt.savefig(img_names[5], bbox_inches='tight'); plt.close()

# =========================================================
# 7. EXPORTACIÓ I FORMATS EXCEL
# =========================================================
output_name = 'Resultats_TFG.xlsx'
with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
    df_res.to_excel(writer, sheet_name='Resum_Executiu', index=False)
    df_analisi.to_excel(writer, sheet_name='Analisis_Economic', index=False)

wb = load_workbook(output_name)
thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
fill_groc, fill_blau, fill_taronja, fill_gris, fill_verd = [PatternFill(start_color=c, end_color=c, fill_type='solid') for c in ['FFF2CC', 'DDEBF7', 'FCE4D6', 'F2F2F2', 'E2EFDA']]

llista_grocs = ['Producció Bruta Teòrica (Gross P50) - Mitjana Anual (MWh/any)', 'Producció Neta Esperada (Net P50) - Mitjana Anual (MWh/any)', 'Producció Neta Conservadora (Net P90) - Mitjana Anual (MWh/any)', 'Hores equivalents a plena potència (h)', 'Factor de Capacitat (%) (Net P50)', 'Factor de capacitat (Gross %)', 'Factor de capacitat (Net 10-year P90 %)']
llista_blaus = ['Ingressos Bruts Anuals - Any 1 (M€/any)', 'OPEX Base (Manteniment Fix) - Any 1 (M€/any)', 'Costos Operatius Totals (OPEX + Peatges) - Any 1 (M€/any)', 'EBITDA (Benefici Brut Operatiu) - Any 1 (M€/any)', 'EBITDA Promig (25 anys) (M€/any)']
llista_taronges = ['Amortització Lineal Anual (M€/any)', 'EBIT (Benefici abans d\'impostos) - Any 1 (M€/any)', 'Impost de Societats Pagat - Any 1 (M€/any)', 'Benefici Net Anual (Comptable) - Any 1 (M€/any)']
llista_grisos = ['Flux de Caixa Lliure (Free Cash Flow) - Any 1 (M€/any)', 'Inversió CAPEX + DEVEX Total (M€)', 'Flux de Caixa Net Acumulat (25 anys) (M€)', 'VAN (Valor Actual Net a FID) (M€)', 'TIR (Taxa Interna de Rendiment) (%)', 'LCOE (€/MWh)', 'Payback Simple des d\'Operació (anys)', 'Payback Descomptat des d\'Operació (anys)']

sense_decimals = [
    'Producció Bruta Teòrica (Gross P50) - Mitjana Anual (MWh/any)', 'Producció Neta Esperada (Net P50) - Mitjana Anual (MWh/any)', 'Producció Neta Conservadora (Net P90) - Mitjana Anual (MWh/any)', 'Hores equivalents a plena potència (h)', 't CO2 equivalent evitats - Any 1', 't CO2 equivalent evitats (Total 25 anys)'
]

ws = wb['Resum_Executiu']
for col, w in zip(['A', 'B', 'C'], [65, 25, 25]): ws.column_dimensions[col].width = w

for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=3):
    m = row[0].value
    color = fill_groc if m in llista_grocs else (fill_blau if m in llista_blaus else (fill_taronja if m in llista_taronges else (fill_gris if m in llista_grisos else (fill_verd if 'CO2' in m else None))))
    for cell in row:
        if color: cell.fill = color
        cell.border = thin_border
        if isinstance(cell.value, (int, float)):
            if cell.col_idx > 1 and m in sense_decimals: cell.number_format = '#,##0'
            else: cell.number_format = '#,##0.000'

ws_econ = wb['Analisis_Economic']
widths_econ = {'A': 8, 'B': 18, 'C': 18, 'D': 18, 'E': 18, 'F': 18, 'G': 18, 'H': 32, 'I': 32, 'J': 34, 'K': 34}
for col, w in widths_econ.items(): ws_econ.column_dimensions[col].width = w

for row in ws_econ.iter_rows(min_row=1, max_row=ws_econ.max_row, min_col=1, max_col=11):
    for cell in row:
        if cell.value is not None:
            cell.border = thin_border
            if cell.row == 1 or cell.value in ['ONSHORE', 'OFFSHORE']: cell.font = Font(bold=True)
            if isinstance(cell.value, (int, float)) and cell.row > 1: cell.number_format = '#,##0.000'

ws_g = wb.create_sheet('Analisi_Grafic')
current_row = 2
row_steps = [28, 23, 23, 28, 34, 60]
for i, img in enumerate(img_names):
    ws_g.add_image(OpenImage(img), f'B{current_row}')
    current_row += row_steps[i]

wb.save(output_name)

# =========================================================
# 8. LÒGICA DE TEXTOS CONDICIONALS DINÀMICS PEL PDF
# =========================================================
from weasyprint import HTML

def fmt(val):
    if isinstance(val, str): return val
    return f"{val:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

def fmt_0(val):
    if isinstance(val, str): return val
    return f"{val:,.0f}".replace(',', 'X').replace('.', ',').replace('X', '.')

# A) Lògica dinàmica energètica
comentari_on_ener = f"El factor de capacitat assolit és del <strong>{fmt(fc_on)}%</strong>. Aquest és un valor excel·lent per a parcs terrestres, indicant una tria molt encertada de l'emplaçament i de la corba de potència." if fc_on >= 40 else f"El factor de capacitat assolit és del <strong>{fmt(fc_on)}%</strong>. Aquest valor és moderat/baix, el que suggereix certes limitacions en el recurs de vent a terra o en l'aerogenerador escollit."

txt_superioritat = " Aquesta dada demostra la superioritat tècnica del recurs al mar respecte al terrestre, acostant la producció eòlica a perfils propis de centrals de càrrega base." if prod_on_p50 < prod_off_p50 else ""
comentari_off_ener = f"El factor de capacitat s'enfila fins al <strong>{fmt(fc_off)}%</strong>.{txt_superioritat}" if fc_off >= 45 else f"El factor de capacitat és de només el <strong>{fmt(fc_off)}%</strong>, indicant que l'emplaçament marítim simulat té un vent menys constant del previst."

# B) Lògica dinàmica econòmica
if van_on > 0 and tir_on > taxa_descompte * 100:
    econ_on_txt = f"El projecte Onshore demostra una robustesa financera aclaparadora. Presenta un <strong>VAN altament positiu de {fmt(van_on)} M€</strong> i una <strong>TIR del {fmt(tir_on)}%</strong>, superant còmodament el cost del capital (WACC de {fmt(taxa_descompte*100)}%) i creant valor net significatiu."
else:
    econ_on_txt = f"El projecte Onshore presenta clars problemes de viabilitat financera en aquest escenari. El seu <strong>VAN és de {fmt(van_on)} M€</strong> i la seva <strong>TIR cau al {fmt(tir_on)}%</strong>, xifres que no assoleixen el llindar mínim requerit i farien inviable la inversió."

if van_off > 0:
    econ_off_txt = f"Malgrat l'alta inversió requerida, el projecte Offshore resulta plenament viable. Aconsegueix un <strong>VAN positiu de {fmt(van_off)} M€</strong> i una TIR del {fmt(tir_off)}%, demostrant que l'escala i el volum d'energia generada superen l'estructura pesada del CAPEX."
else:
    econ_off_txt = f"La construcció al mar genera barreres econòmiques molt dures amb una inversió inicial de <strong>{fmt(cap_off_total)} M€</strong>. Com a conseqüència, el projecte tanca amb un <strong>VAN negatiu de {fmt(van_off)} M€</strong> i una TIR insuficient del {fmt(tir_off)}%, destruint capital net."

comp_econ_txt = f"És fonamental remarcar que <strong>el projecte que té el millor VAN i TIR és indubtablement el millor a nivell financer</strong>. En aquesta simulació, l'Onshore és l'opció preferent en eficiència de capital." if van_on > van_off else f"És fonamental remarcar que <strong>el projecte que té el millor VAN i TIR és indubtablement el millor a nivell financer</strong>. En aquesta simulació, la tecnologia Offshore s'imposa clarament com la millor inversió."

# C) Lògica dinàmica LCOE (factors clau)
if lcoe_on < ppa_on_adj:
    lcoe_on_txt = f"Això deixa un marge de benefici ampli i segur front al preu del mercat elèctric."
else:
    lcoe_on_txt = f"Això suposa un risc sever, ja que el cost de generació supera o iguala el preu de venda esperat."

if lcoe_off < ppa_off_adj:
    lcoe_off_txt = f"Aquest valor permet tenir un marge positiu considerant el preu de venda de l'energia previst a l'inici de l'operació."
else:
    lcoe_off_txt = f"Això fa que sigui completament impossible competir sense un contracte especial o subsidi de l'Estat, atès que costa més produir l'energia que el que s'ingressa per la seva venda."

pay_on_status = f"favorable, amb {fmt(p_s_on)} anys per al simple (inferior al límit crític de 10 anys)" if (isinstance(p_s_on, (int, float)) and p_s_on < 10) else f"desfavorable/risc alt, triga {fmt(p_s_on)} anys (per sobre del desitjat de 10 anys)"
pay_off_status = f"favorable, recuperant la inversió en {fmt(p_s_off)} anys" if (isinstance(p_s_off, (int, float)) and p_s_off < 10) else f"desfavorable, allargant-se fins als {fmt(p_s_off)} anys, i un descomptat que se situa en {fmt(p_d_off)} anys"

# D) Lògica dinàmica Conclusions
if van_on > 0 and van_off <= 0:
    conclusions_txt = f"Després de comparar ambdues tecnologies, la tecnologia guanyadora indiscutible és l'<strong>Onshore</strong>. Això és degut al fet que presenta, simultàniament, un menor LCOE ({fmt(lcoe_on)} €/MWh), un VAN altament elevat i positiu, i el Payback més ràpid, mentre que l'Offshore queda completament descartat."
elif van_off > 0 and van_on <= 0:
    conclusions_txt = f"Després d'avaluar l'impacte complet dels fluxos de caixa, la tecnologia guanyadora és l'<strong>Offshore</strong>. El projecte terrestre és incapaç de sobreviure, mentre que el marí aconsegueix vèncer el seu CAPEX inicial i registrar un VAN positiu dominant."
elif van_on > 0 and van_off > 0:
    if van_on > van_off:
        conclusions_txt = f"Ambdós projectes són rendibles en termes absoluts, però la tecnologia guanyadora és l'<strong>Onshore</strong> per presentar un VAN més elevat ({fmt(van_on)} M€) i un termini de recuperació (Payback) molt més curt i segur davant de volatilitats del mercat."
    else:
        conclusions_txt = f"Ambdós projectes són viables, però la tecnologia guanyadora final és l'<strong>Offshore</strong>, ja que genera un major volum net de riquesa (VAN de {fmt(van_off)} M€) al llarg de la seva explotació, compensant el fet de tenir un Payback més llarg."
else:
    conclusions_txt = "<strong>Conclusió Crítica:</strong> Cap de les dues tecnologies és viable sota els paràmetres de mercat lliure actuals. Ambdós projectes registren un VAN negatiu i, per tant, s'haurien de rebutjar immediatament a l'espera de subsidis, preus d'energia (PPA) més alts o reduccions dràstiques de CAPEX."

with open(img_names[3], "rb") as f: img_prod = base64.b64encode(f.read()).decode('utf-8')
with open(img_names[4], "rb") as f: img_cf = base64.b64encode(f.read()).decode('utf-8')
with open(img_names[5], "rb") as f: img_rad = base64.b64encode(f.read()).decode('utf-8')

# HTML definitiu amb text-align: justify !important a tot arreu
html_content = f"""
<!DOCTYPE html>
<html lang="ca">
<head>
    <meta charset="UTF-8">
    <style>
        @page {{ size: A4; margin: 25mm 20mm; }}
        body, p, li, div, span {{ font-family: 'Calibri', 'Arial', sans-serif; font-size: 12pt; text-align: justify !important; line-height: 1.5; color: #000000; }}
        h1 {{ font-family: 'Arial', sans-serif; font-size: 18pt; text-align: center !important; color: #1a426b; margin-bottom: 30px; text-transform: uppercase; }}
        h2 {{ font-family: 'Arial', sans-serif; font-size: 14pt; text-align: left !important; color: #1a426b; border-bottom: 2px solid #1a426b; padding-bottom: 4px; margin-top: 30px; margin-bottom: 15px; page-break-after: avoid; }}
        h3 {{ font-family: 'Arial', sans-serif; font-size: 12pt; text-align: left !important; color: #b35900; margin-top: 15px; margin-bottom: 10px; page-break-after: avoid; }}
        p {{ margin-bottom: 12px; }}
        ul {{ margin-bottom: 15px; padding-left: 20px; }}
        li {{ margin-bottom: 8px; }}
        .metric-box {{ background-color: #f4f6f9; padding: 12px 15px; border-left: 4px solid #1a426b; margin-bottom: 15px; font-size: 11pt; page-break-inside: avoid; text-align: justify !important; }}
        .img-container {{ text-align: center !important; margin: 20px 0; page-break-inside: avoid; }}
        .img-container img {{ max-width: 100%; height: auto; border: 1px solid #ddd; }}
    </style>
</head>
<body>

    <h1>Comparativa Tecnologia Energia Eòlica Onshore vs Offshore</h1>

    <h2>1. Anàlisi Energètica</h2>

    <h3>Tecnologia Onshore (Terrestre)</h3>
    <p>L'avaluació del recurs eòlic terrestre revela uns paràmetres de distribució de Weibull calculats en el model. Això es tradueix en una <strong>Producció Neta Esperada (Net P50) de {fmt_0(prod_on_p50)} MWh/any</strong> i unes hores equivalents a plena potència de {fmt_0(h_eq_on)} h.</p>
    <div class="metric-box">
        <strong>Anàlisi del recurs:</strong> {comentari_on_ener}
    </div>

    <h3>Tecnologia Offshore (Marina)</h3>
    <p>A l'entorn marí, s'ha analitzat igualment el perfil del vent a l'emplaçament objectiu. La <strong>Producció Neta Esperada (Net P50) aconseguida és de {fmt_0(prod_off_p50)} MWh/any</strong>, aconseguint treballar l'equivalent a {fmt_0(h_eq_off)} hores a plena càrrega anuals.</p>
    <div class="metric-box">
        <strong>Anàlisi del recurs:</strong> {comentari_off_ener}
    </div>

    <h3>Comparativa Energètica</h3>
    <p>A nivell purament energètic, la tecnologia que presenta una <strong>Net P50 més alta és la que major energia global produeix per a la viabilitat de l'estudi</strong>. El gràfic inferior mostra la distribució esperada d'aquesta producció de forma sostinguda mes a mes durant l'any 2025.</p>

    <div class="img-container">
        <img src="data:image/png;base64,{img_prod}" alt="Gràfic Producció Mensual">
    </div>

    <h2>2. Anàlisi Econòmica</h2>

    <h3>Tecnologia Onshore</h3>
    <p>El cost d'instal·lació total (CAPEX + DEVEX) es fixa en <strong>{fmt(cap_on_total)} M€</strong>. Un cop en funcionament, l'operació i el manteniment el primer any es comporten d'acord amb el pressupost d'OPEX previst. {econ_on_txt}</p>

    <h3>Tecnologia Offshore</h3>
    <p>L'explotació marina enfronta reptes logístics complexos, el que suposa un cost d'instal·lació complet de <strong>{fmt(cap_off_total)} M€</strong>. {econ_off_txt}</p>

    <h3>Comparativa Econòmica</h3>
    <p>{comp_econ_txt} Cal tenir present que el projecte amb millors mètriques reflecteix de manera rigorosa l'aportació neta de capital reajustada amb el pas del temps.</p>

    <div class="img-container" style="margin-bottom: 40px;">
        <img src="data:image/png;base64,{img_cf}" alt="Flux de Caixa Acumulat">
    </div>

    <div class="img-container">
        <img src="data:image/png;base64,{img_rad}" alt="Radiografia Economica">
    </div>

    <h2>3. Anàlisi de Factors Claus</h2>

    <p>El principal element diferenciador en projectes renovables és el <strong>LCOE (Levelized Cost of Energy)</strong>. Aquesta mètrica defineix el preu d'equilibri per fabricar un MWh al llarg de la vida útil de la planta.</p>
    <ul>
        <li><strong>Onshore:</strong> Presenta un LCOE de <strong>{fmt(lcoe_on)} €/MWh</strong>. {lcoe_on_txt} Pel que fa al seu retorn, el termini es considera {pay_on_status}, amb un Payback Descomptat final de {fmt(p_d_on)} anys.</li>
        <li><strong>Offshore:</strong> El seu LCOE és de <strong>{fmt(lcoe_off)} €/MWh</strong>. {lcoe_off_txt} En termes de risc i recuperació de caixa, el termini és {pay_off_status}.</li>
    </ul>

    <h2>4. Conclusions</h2>

    <p>{conclusions_txt}</p>

    <div class="metric-box">
        <strong>Reflexió sobre el pes de les mètriques (VAN vs Payback):</strong><br>
        En la disciplina del <em>Project Finance</em>, hi ha consens que el <strong>VAN (Valor Actual Net) és el criteri decisiu absolut</strong>, ja que mesura la riquesa real i tangible que el projecte aportarà als seus desenvolupadors. Tot i això, el Payback juga un paper crucial com a eina de control de risc: com abans es recuperen els diners, menys exposició es té a incerteses futures.<br><br>
        En cas de discrepàncies entre dos projectes rendibles, un Payback ràpid pot tenir prou pes com per decantar la balança si l'inversor vol minimitzar el risc, actuant com a factor mitigador en la presa de decisions corporatives.
    </div>

</body>
</html>
"""

pdf_name = "Comparativa_Eolica_Onshore_vs_Offshore.pdf"
HTML(string=html_content).write_pdf(pdf_name)

# =========================================================
# 9. EMPAQUETAR I DESCARREGAR ARXIU ZIP
# =========================================================
for img in img_names: os.remove(img)

zip_name = "Resultats_Estudi_Eolic.zip"
with zipfile.ZipFile(zip_name, 'w') as zipf:
    zipf.write(output_name)
    zipf.write(pdf_name)

print("✅ Procés finalitzat: Excel i PDF generats.")

if colab_env:
    files.download(zip_name)

✅ Procés finalitzat: Excel i PDF generats.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>